# Embedding compression + pair-feature assembly

Trains a small undercomplete autoencoder that compresses the 4096-dim Qwen-3
embedding down to 512 dims, then uses the bottleneck representation to build
the per-pair feature table consumed by `cor_modelling.ipynb` and (legacy) by
the older DNN. The Siamese DNN bypasses the bottleneck and reads
`embeddings_raw.parquet` directly.


In [1]:
# --- 1. Train the embedding autoencoder ---
# Linear bottleneck (no activation on the second encoder layer) so distances
# in the compressed space stay approximately Euclidean - this matters because
# we use it for cosine similarity downstream.

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import polars as pl

# ---- Config ----
emb_model = "qwen3-8b-with-instruction"
path      = f"../../data/raw/{emb_model}/embeddings_raw.parquet"
n_dims    = 512

# ---- Load embeddings ----
item_embeddings = pl.read_parquet(path)
n_embeddings    = len([col for col in item_embeddings.columns if col.startswith("emb")])

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
X_train  = item_embeddings.select(pl.col("^emb.*$")).to_numpy()
X_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)


# ---- Architecture ----
class PsychometricAE(nn.Module):
    def __init__(self, input_dim=n_embeddings, bottleneck_dim=n_dims):
        super().__init__()
        # Encoder funnels into the linear bottleneck
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, bottleneck_dim),
        )
        # Decoder mirrors the encoder; only present during training
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


# ---- Train ----
model     = PsychometricAE().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Training autoencoder...")
for epoch in range(500):
    optimizer.zero_grad()
    loss = criterion(model(X_tensor), X_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/500], Loss: {loss.item():.6f}")

# ---- Pull out the compressed representation ----
with torch.no_grad():
    compressed_vectors = model.encoder(X_tensor).cpu().numpy()

column_names = [f"emb_{i+1}" for i in range(n_dims)]
embeddings_compressed = (
    item_embeddings
    .with_columns(pl.Series("compressed_embedding", [list(v) for v in compressed_vectors]))
    .select([
        pl.col("item"),
        pl.col("compressed_embedding").list.to_struct(fields=column_names),
    ])
    .unnest("compressed_embedding")
)
print("Success! Compressed features added to dataframe.")


Training Autoencoder...
Epoch [20/500], Loss: 0.000124
Epoch [40/500], Loss: 0.000119
Epoch [60/500], Loss: 0.000109
Epoch [80/500], Loss: 0.000096
Epoch [100/500], Loss: 0.000085
Epoch [120/500], Loss: 0.000075
Epoch [140/500], Loss: 0.000069
Epoch [160/500], Loss: 0.000065
Epoch [180/500], Loss: 0.000061
Epoch [200/500], Loss: 0.000057
Epoch [220/500], Loss: 0.000055
Epoch [240/500], Loss: 0.000052
Epoch [260/500], Loss: 0.000050
Epoch [280/500], Loss: 0.000047
Epoch [300/500], Loss: 0.000047
Epoch [320/500], Loss: 0.000044
Epoch [340/500], Loss: 0.000043
Epoch [360/500], Loss: 0.000042
Epoch [380/500], Loss: 0.000040
Epoch [400/500], Loss: 0.000040
Epoch [420/500], Loss: 0.000038
Epoch [440/500], Loss: 0.000038
Epoch [460/500], Loss: 0.000036
Epoch [480/500], Loss: 0.000038
Epoch [500/500], Loss: 0.000035
Success! Compressed features added to dataframe.


In [2]:
# --- 2. Persist the trained weights ---
# Reused at holdout time (holdout_evaluation.ipynb) and inside
# dnn_siamese_holdout_eval.ipynb to recompute global_sim consistently.

torch.save(model.state_dict(), "../../models/psychometric_ae_weights.pt")
print("Model saved successfully!")


Model saved successfully!


In [3]:
# --- 3. Load pair table + item-level features and join in the compressed embeddings ---

import os

item_cors_df = pl.read_parquet("../../data/processed/item_correlations_with_cross.parquet")
item_list_df = pl.read_parquet("../../data/processed/item_list_sentiment.parquet")

# Item-level features get joined to the item list (handy for inspection)
items_embedded = item_list_df.join(embeddings_compressed, on="item", how="left")

os.makedirs(f"../../data/clustered_embeddings/{emb_model}", exist_ok=True)
items_embedded.write_parquet(
    f"../../data/clustered_embeddings/{emb_model}/item_list_emb_{n_dims}.parquet"
)


In [4]:
# --- 4. Attach per-pair embeddings to the pair table ---
# Two joins (Parameter1 + Parameter2) so each row carries both items' vectors.
# Drop the per-item descriptive stats columns to avoid leakage into the pair model.

item_cors_df = (
    item_cors_df
    .join(embeddings_compressed, right_on="item", left_on="Parameter1", how="left", suffix="_item1")
    .join(embeddings_compressed, right_on="item", left_on="Parameter2", how="left", suffix="_item2")
)

item_list_df = item_list_df.select(
    pl.exclude(["mean", "sd", "max", "min", "item_right"])
)


In [5]:
# --- 5. Derive geometric pair features ---
# prod_i: element-wise product of the two compressed embeddings - a per-dim
#         analog to cosine similarity that the model can up/down-weight.
# global_sim: a single cosine similarity over the full 512-dim vector.

import polars.selectors as cs

cols_1 = [f"emb_{i+1}"        for i in range(n_dims)]
cols_2 = [f"emb_{i+1}_item2"  for i in range(n_dims)]

product_features = [
    (pl.col(c1) * pl.col(c2)).alias(f"prod_{i+1}")
    for i, (c1, c2) in enumerate(zip(cols_1, cols_2))
]

dot_product = pl.sum_horizontal([pl.col(c1) * pl.col(c2) for c1, c2 in zip(cols_1, cols_2)])
norm_a      = pl.sum_horizontal([pl.col(c) ** 2 for c in cols_1]).sqrt()
norm_b      = pl.sum_horizontal([pl.col(c) ** 2 for c in cols_2]).sqrt()

# Drop raw emb columns once derived features are built (prod_i + global_sim
# carry all the inter-item information the model needs)
df_final = (
    item_cors_df
    .with_columns(product_features)
    .with_columns(global_sim = dot_product / (norm_a * norm_b))
    .select(~cs.starts_with("emb"))
)

# Suffix item-level features so the two sides survive the join with distinct names
item_list_v1 = item_list_df.rename(lambda c: c + "_item1")
item_list_v2 = item_list_df.rename(lambda c: c + "_item2")

df_final = (
    df_final
    .join(item_list_v1, left_on="Parameter1", right_on="item_item1", how="left")
    .join(item_list_v2, left_on="Parameter2", right_on="item_item2", how="left")
)


In [6]:
# Quick sanity-check of the final pair-level feature table
df_final.head()


Parameter1,Parameter2,r,pair_negative,pair_positive,contradiction,entail,logic_neutral,similarity,thematic_intensity,logical_friction,sentiment_balance,prod_1,prod_2,prod_3,prod_4,prod_5,prod_6,prod_7,prod_8,prod_9,prod_10,prod_11,prod_12,prod_13,prod_14,prod_15,prod_16,prod_17,prod_18,prod_19,prod_20,prod_21,prod_22,prod_23,prod_24,prod_25,…,prod_501,prod_502,prod_503,prod_504,prod_505,prod_506,prod_507,prod_508,prod_509,prod_510,prod_511,prod_512,global_sim,sent_positive_item1,sent_neutral_item1,sent_negative_item1,emo_neutral_item1,emo_surprise_item1,emo_joy_item1,emo_fear_item1,emo_anger_item1,emo_sadness_item1,emo_disgust_item1,top_sentiment_item1,top_emotion_item1,sent_positive_item2,sent_neutral_item2,sent_negative_item2,emo_neutral_item2,emo_surprise_item2,emo_joy_item2,emo_fear_item2,emo_anger_item2,emo_sadness_item2,emo_disgust_item2,top_sentiment_item2,top_emotion_item2
str,str,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""Do you often long for exciteme…","""Do you often need understandin…",0.201111,0.011093,0.679199,0.999023,0.000023,0.001013,0.015961,3.6721e-7,0.015945,0.668106,9.3386e-7,0.000275,0.001339,-0.000022,0.000183,-0.000018,0.001153,0.000002,-0.000203,0.000176,0.000225,-0.00036,0.000199,-0.000022,0.001651,0.000036,-0.000101,0.000316,0.000455,0.000071,-0.000034,-0.00001,0.000448,0.000117,-0.000168,…,0.000131,-0.000015,-0.000072,0.000101,0.000114,-0.000447,-0.000133,-0.000116,-0.000139,-0.000193,0.001858,0.000977,0.47997,0.783163,0.205401,0.011436,0.784398,0.119589,0.032254,0.030117,0.02025,0.01002,0.003373,"""positive""","""neutral""",0.597145,0.379198,0.023657,0.749233,0.072289,0.127554,0.005687,0.017466,0.013736,0.014034,"""positive""","""neutral"""
"""Do you often long for exciteme…","""Do you stop and think things o…",-0.013197,0.02562,0.254639,1.0,0.000011,0.000136,0.006691,7.2584e-8,0.006691,0.229019,-0.000008,0.000564,0.000881,-0.000238,-0.00014,0.000024,-0.000417,-0.000012,0.000141,0.000336,-0.000094,0.000298,0.000001,-0.000024,-0.000246,0.000045,-0.000976,-0.000164,-0.000404,0.001151,0.000004,0.000023,0.00065,0.000349,0.000528,…,-0.000143,0.000038,-0.000775,-0.000327,-0.00012,0.000264,0.00014,0.000346,-0.000031,0.00005,0.001631,-0.000239,0.392,0.783163,0.205401,0.011436,0.784398,0.119589,0.032254,0.030117,0.02025,0.01002,0.003373,"""positive""","""neutral""",0.021467,0.800833,0.1777,0.062341,0.01233,0.002852,0.072813,0.784729,0.011009,0.053926,"""neutral""","""anger"""
"""Do you often long for exciteme…","""If you say you will do somethi…",-0.025196,0.038971,0.341797,0.012024,0.000207,0.987793,0.002369,4.8940e-7,0.000028,0.302826,-0.000015,0.000369,0.001211,-0.000212,0.000138,-0.000013,-0.001215,0.000161,-0.00015,0.000302,0.000035,0.000707,0.000056,0.000083,-0.000053,0.000075,-0.00034,0.000059,-0.000017,0.000398,-0.000033,0.000045,0.00058,-0.000337,0.000429,…,-0.000035,0.000004,-0.000045,-0.000108,0.000097,0.000277,0.000046,-0.000304,0.000051,-0.000179,0.00034,0.000051,0.372883,0.783163,0.205401,0.011436,0.784398,0.119589,0.032254,0.030117,0.02025,0.01002,0.003373,"""positive""","""neutral""",0.052955,0.69619,0.250854,0.51845,0.006533,0.004173,0.020429,0.382281,0.01385,0.054285,"""neutral""","""neutral"""
"""Do you often long for exciteme…","""Do your moods go up and down?""",0.135682,0.02153,0.2771,0.974609,0.001226,0.024017,0.02037,0.000025,0.019853,0.255569,0.000017,0.000286,0.001272,-0.000228,0.000257,-0.000017,0.000454,-0.000007,-0.000043,0.000594,0.000086,0.000356,-0.000122,-0.000047,0.000353,0.000052,-0.000475,0.000384,-0.00031,0.000199,-0.00032,0.000042,-0.000592,-0.000044,0.000677,…,0.000172,-0.000033,0.000072,-0.00001,-0.00006,0.000588,-0.000145,0.000251,0.000074,0.000353,0.001473,0.000057,0.516564,0.783163,0.205401,0.011436,0.784398,0.119589,0

In [7]:
# --- 6. Save final pair-level feature table ---
df_final.write_parquet(
    f"../../data/clustered_embeddings/{emb_model}/autoencoded_clusters_{n_dims}.parquet"
)
